# 全库逐边闭环审计
权威提交 3ec62c55c8e3433854246ea50f889668b2659cc4；以下检查读取冻结的 TaskSpec 与台账，不访问远端、不启动 QM。
分类判断参见 decisions.json 与 README.md；核验数值存在不等于重新完成化学验收。

In [1]:
import hashlib
import json
from collections import Counter
from pathlib import Path

root = Path.cwd()
if not (root / "all_edges.json").exists():
    root = root / "gold_compute/library_audit_20261002"
ledger = json.loads((root / "all_edges.json").read_text())
manifest = json.loads((root / "source_manifest.json").read_text())
for item in manifest:
    blob = (root / item["snapshot"]).read_bytes()
    assert len(blob) == item["bytes"]
    assert hashlib.sha256(blob).hexdigest() == item["sha256"]
tasks = [json.loads(p.read_text()) for p in (root / "source_snapshot/published/ideapool/tasks").glob("*/*.json")]
edges = [e for t in tasks for e in t["gold"]["mechanism_graph"]["edges"]]
assert len(tasks) == 52 and len(edges) == 81
assert {e["id"] for e in edges} == {r["edge_id"] for r in ledger}
assert len({r["edge_id"] for r in ledger}) == 81
assert sum(e["barrier_kcal_mol"] is not None for e in edges) == 57
assert sum(r["status"] == "open" for r in ledger) == 10
Counter(r["status"] for r in ledger)

Counter({'backfilled': 57, 'open': 10, 'inaccessible_ts': 8, 'non_barrier_property': 4, 'waived': 2})

In [2]:
[(r["edge_id"], r["short_name"], r["reason"]) for r in ledger if r["status"] == "open"]

[('HI1_E1', '脯氨酸烯胺缩合', '本轮只验收两条 C–C 加成，E1 未计算且撤回旧 19.2；属于本轮范围外/暂留缺口，没有永久放弃记录。'), ('HI1_E3', 'iminium-aldolate 水解', '本轮范围外且撤回旧 16.9；未计算不能等同明确永久放弃。'), ('HL1_E1', 'Beckmann：肟 OH 质子化', '最新源保持 null；旧 Gaussian 的无鞍点扫描与 ORCA 错位 PT 候选均不是现行合格回填。未见正式豁免/放弃。'), ('NI5_E1', '水溶液亚胺形成：C–N 加成', '最新题源明确写尚无无约束 TS 能垒、暂缺测；未见明确放弃。'), ('NR1_E1', '频哪醇 OH 质子化', '最新源写 PT 未确立非零垒；旧 Gaussian 约束扫描的 0 不是当前合格回填或正式豁免。分子内互变规则不自动适用于本分子间酸活化。'), ('NR1_E2', '质子化频哪醇脱水', '最新源明确未找到独立鞍点，参考垒缺测；搜索失败不是已明确放弃。'), ('NU5_E2', 'Passerini：甲酸根捕获腈鎓', '最新题源明确捕获能垒缺测；旧四步图的 E2/E3 编号及 8.2/0 不可移植到当前三步图。'), ('NY3_E1', 'Na⁺ 模型：OH⁻ 加成', '用户改为 Na⁺ 制衡中性模型后明确要求重算全图；裸离子旧值作废，未见新模型闭环或放弃记录。 9/26 复盘记录有限 TS/扫描尝试失败及机理耦合疑点，预算耗尽、待科学裁决不是正式放弃。'), ('NY3_E2', 'Na⁺ 模型：甲氧基消除', '同上；裸 OH⁻ 旧 12.5 不能回填 Na⁺ 新模型。 9/26 复盘记录有限 TS/扫描尝试失败及机理耦合疑点，预算耗尽、待科学裁决不是正式放弃。'), ('NY3_E3', 'Na⁺ 模型：甲酸→甲氧基 PT', '同上；该边虽为 PT，源明确三步待 Na 模型战役回填；旧裸离子 0.0 及分子内互变豁免均不自动关闭此边。 9/26 复盘记录有限 TS/扫描尝试失败及机理耦合疑点，预算耗尽、待科学裁决不是正式放弃。')]